In [ ]:
import functools
import hashlib
import itertools
from collections.abc import Mapping
from pathlib import Path

import numpy as np
import numpy.typing as npt
import seaborn as sns
import torch
import xarray as xr
from bonner.caching import cache
from bonner.computation.metrics import pearson_r
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from PIL import Image
from sklearn.linear_model import RidgeCV
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from tqdm.auto import tqdm

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    split_by_repetition,
)
from lib.spectra import (
    CrossDecomposition,
    compute_spectra_with_n_fold_cross_validation,
    plot_spectra,
)
from lib.utilities import (
    JOURNAL_MATPLOTLIBRC,
    apply_gabor_filter_bank,
    bin_data,
    create_gabor_filter_bank,
    extract_geometrically_spaced_bins,
    mathtext_exponent_label,
)


def dict_product(parameters: Mapping):
    return (
        dict(zip(parameters.keys(), x, strict=False))
        for x in itertools.product(*parameters.values())
    )


def fit_linear_ridge_regression(
    x: xr.DataArray,
    y: xr.DataArray,
    /,
    *,
    alphas: npt.NDArray[np.floating],
) -> RidgeCV:
    hash_ = hashlib.blake2b(alphas.tobytes(), digest_size=4).hexdigest()
    cacher = cache(
        f"ridge_regression/x={x.name}/y={y.name}/alphas={hash_}.pkl",
    )
    return cacher(_fit_linear_ridge_regression)(
        x,
        y,
        alphas=alphas,
    )


def _fit_linear_ridge_regression(
    x: xr.DataArray,
    y: xr.DataArray,
    /,
    *,
    alphas: npt.NDArray[np.floating],
) -> RidgeCV:
    linear_regression = RidgeCV(alphas=alphas, alpha_per_target=True)
    linear_regression.fit(x.to_numpy(), y.to_numpy())
    return linear_regression


FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

ALPHAS = np.geomspace(1e4, 1e8, num=30)

## Gabor models

In [ ]:
stimulus_set = nsd.StimulusSet()

SIZE = stimulus_set.stimuli.sizes["height"]

transform = v2.Compose([
    v2.Grayscale(),
    v2.ToDtype(torch.float32, scale=True),
])


def collate_fn(batch: list[Image.Image]) -> torch.Tensor:
    batch_ = np.stack(batch, axis=0).transpose(0, 3, 1, 2)
    return transform(torch.from_numpy(batch_))


dataloader = DataLoader(
    dataset=stimulus_set,
    batch_size=1024,
    collate_fn=collate_fn,
)


In [ ]:
# https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1006897

parameter_ranges = {
    "x": np.linspace(-0.45, 0.45, num=8),
    "y": np.linspace(-0.45, 0.45, num=8),
    "aspect_ratio": [1],
    "orientation": np.linspace(0, np.pi, num=8),
    "phase": [0, np.pi / 2],
    "scale": [0.025, 0.05, 0.075],
}

parameters = []
for parameters_ in dict_product(parameter_ranges):
    frequencies = np.arange(1, 4) / (4 * parameters_["scale"])
    for frequency in frequencies:
        parameter_combination = parameters_ | {"frequency": frequency}
        parameters.append(parameter_combination)


filter_bank = create_gabor_filter_bank(
    size=SIZE,
    parameters=parameters,
)

simple_cell_responses = (
    apply_gabor_filter_bank(
        filter_bank,
        stimulus_set=stimulus_set,
        dataloader=dataloader,
    )
    .set_xindex("phase")
    .load()
)

complex_cell_responses = (
    xr.concat(
        [
            simple_cell_responses.sel(phase=phase)
            .drop_vars("phase")
            .expand_dims(phase=[phase])
            for phase in [0, np.pi / 2]
        ],
        dim="phase",
    )
    ** 2
).sum("phase")

complex_cell_responses = complex_cell_responses.assign_coords(
    {"phase": ("filter", np.full((complex_cell_responses.sizes["filter"],), np.nan))},
)

gabor_model_responses = (
    xr.concat(
        [
            simple_cell_responses.drop_indexes("phase"),
            complex_cell_responses,
        ],
        dim="filter",
    )
    .set_xindex([*parameter_ranges, "frequency"])
    .rename(simple_cell_responses.name)
)

gabor_model_responses = gabor_model_responses.rename({
    "stimulus": "presentation",
    "filter": "neuroid",
}).assign_coords({"stimulus": ("presentation", np.arange(len(stimulus_set)))})
gabor_model_responses -= gabor_model_responses.mean("presentation")
gabor_model_responses /= gabor_model_responses.std("presentation")
gabor_model_responses = gabor_model_responses.rename(
    f"{gabor_model_responses.name}.z_score=True",
)

## neural data

In [ ]:
bin_edges, bin_centers = extract_geometrically_spaced_bins(
    start=1,
    stop=10_000,
    density=3,
)

REFERENCE_SUBJECT = 0
ROIS = ("V1", "V2", "V3", "V4")

datasets = {
    roi: {
        subject: nsd.load_dataset(
            subject=subject,
            roi=roi,
            preprocessing="fithrf",
            z_score=True,
        )
        for subject in range(2)
    }
    for roi in ROIS
}

all_stimuli = compute_shared_stimuli([datasets["V1"][0]], n_repetitions=3)
shared_stimuli = compute_shared_stimuli(datasets["V1"].values(), n_repetitions=3)

## direct model-to-brain spectral comparisons

In [ ]:
datasets_direct = {
    roi: split_by_repetition(
        filter_by_stimulus(
            dataset[REFERENCE_SUBJECT],
            stimuli=all_stimuli,
        ),
        n_repetitions=3,
    )
    for roi, dataset in datasets.items()
}

gabor_model_responses_ = filter_by_stimulus(
    gabor_model_responses,
    stimuli=all_stimuli,
)

spectra = []
spectra_unbinned = []
for roi in tqdm(ROIS, desc="region of interest", leave=False):
    spectra_ = compute_spectra_with_n_fold_cross_validation(
        x_train=datasets_direct[roi][0],
        y_train=gabor_model_responses_,
        x_test=datasets_direct[roi][0],
        y_test=gabor_model_responses_,
        n_folds=8,
        n_permutations=5_000,
    )
    spectra_unbinned.append(
        spectra_["covariance"].expand_dims({
            "region of interest": [roi],
            "comparison": ["brain-model"],
        })
    )
    spectra_ = bin_data(
        spectra_,
        bin_edges={"component": bin_edges},
        bin_centers={"component": bin_centers},
        dim="rank",
    ).expand_dims({
        "region of interest": [roi],
        "comparison": ["brain-model"],
    })
    spectra.append(spectra_)

    spectra_ = compute_spectra_with_n_fold_cross_validation(
        x_train=datasets_direct[roi][0],
        y_train=datasets_direct[roi][1],
        x_test=datasets_direct[roi][0],
        y_test=datasets_direct[roi][1],
        n_folds=8,
        n_permutations=5_000,
    )
    spectra_unbinned.append(
        spectra_["covariance"].expand_dims({
            "region of interest": [roi],
            "comparison": ["within-subject"],
        })
    )
    spectra_ = bin_data(
        spectra_,
        bin_edges={"component": bin_edges},
        bin_centers={"component": bin_centers},
        dim="rank",
    ).expand_dims({
        "region of interest": [roi],
        "comparison": ["within-subject"],
    })
    spectra.append(spectra_)

spectra_unbinned = xr.merge(spectra_unbinned)
spectra = xr.merge(spectra)

In [ ]:
fig, axes = plt.subplots(
    ncols=len(ROIS),
    nrows=1,
    figsize=(6, 3),
    sharex=True,
    sharey=True,
)
for ax, roi in zip(axes.flat, ROIS, strict=True):
    plot_spectra(
        spectra.sel({"region of interest": roi}),
        ax=ax,
        hue="comparison",
        metric="covariance",
        hue_order=["brain-model", "within-subject"],
        hide_insignificant=True,
        null_quantile=0.999,
        palette=["mediumvioletred", "dimgray"],
    )
    ax.set_title(roi)

ax = axes[0]
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-8, top=1e-1)
ax.set_xticks([1, 1e1, 1e2, 1e3])

ytick_exponents = list(range(-8, 0))
ax.set_yticks(
    [10**exponent for exponent in ytick_exponents],
    labels=[
        mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
        for exponent in ytick_exponents
    ],
)
ax.legend(borderaxespad=0.25, borderpad=0.25, handletextpad=0.05)

fig.supxlabel("rank", x=0.55)
fig.supylabel("covariance", y=0.53)
fig.suptitle(
    "shared variance between Gabor filter bank features and cortical representations",
)
fig.set_facecolor("w")

In [ ]:
fig, axes = plt.subplots(
    ncols=len(ROIS),
    nrows=1,
    figsize=(6, 2.75),
    sharex=True,
    sharey=True,
)
for ax, roi in zip(axes.flat, ROIS, strict=True):
    spectra_ = spectra_unbinned["covariance"].sel({"region of interest": roi})

    # this do be kinda sus, but this is what Stringer et al. (2019) does *confused shrug*
    spectra_ = spectra_.where(spectra_ > 0, 0)
    spectra_ /= spectra_.sum("component")

    sns.lineplot(
        ax=ax,
        data=spectra_.cumsum("component").to_dataframe().reset_index(),
        x="component",
        y="covariance",
        hue="comparison",
        hue_order=["brain-model", "within-subject"],
        palette=["mediumvioletred", "dimgray"],
        legend=roi == "V1",
        errorbar="sd",
    )
    ax.set_title(roi)
    ax.set_xlabel("")

ax = axes[0]
ax.set_xscale("log")
ax.set_ylim(bottom=0)
ax.set_xlim(left=1, right=2e3)
ax.set_xticks([1, 1e1, 1e2, 1e3])
ax.set_ylabel("cumulative covariance\n(normalized)")
ax.legend()

fig.supxlabel("rank", x=0.55, y=0.05)
fig.set_facecolor("w")

# save_figure(fig, filepath=FIGURES_HOME / "gabor-model.pdf")

## linear encoding models

In [ ]:
n_train = len(all_stimuli) // 2
stimuli = {
    "train": set(sorted(all_stimuli)[:n_train]),
    "test": set(sorted(all_stimuli)[n_train:]),
}

datasets_encoding_within = {
    subset: {
        roi: split_by_repetition(
            filter_by_stimulus(
                dataset[REFERENCE_SUBJECT],
                stimuli=stimuli_,
            ),
            n_repetitions=3,
        )
        for roi, dataset in datasets.items()
    }
    for subset, stimuli_ in stimuli.items()
}

gabor_model_responses_within = {
    subset: filter_by_stimulus(gabor_model_responses, stimuli=stimuli_)
    for subset, stimuli_ in stimuli.items()
}

### within-subject

In [ ]:
z_score = False

predicted_responses = {}
for roi in tqdm(ROIS, desc="region of interest", leave=False):
    predicted_responses[roi] = {}
    for repetition in tqdm((0, 1), desc="repetition", leave=False):
        linear_regression = fit_linear_ridge_regression(
            gabor_model_responses_within["train"],
            datasets_encoding_within["train"][roi][repetition],
            alphas=ALPHAS,
        )

        z_ = datasets_encoding_within["test"][roi][repetition].copy()
        z_ = z_.rename(f"{z_.name}.predicted.ridge")
        z_.data = linear_regression.predict(
            gabor_model_responses_within["test"].to_numpy(),
        )

        if z_score:
            z_ = (z_ - z_.mean("presentation")) / z_.std("presentation")
            z_ = z_.rename(f"{z_.name}.z_score=True")

        predicted_responses[roi][repetition] = z_.astype(np.float32)


In [ ]:
n_voxels = 8
fig, axes = plt.subplots(figsize=(12, 3), ncols=4, nrows=2, sharex=True, sharey=True)

for i_voxel, ax in zip(range(n_voxels), axes.flat, strict=True):
    ax.scatter(
        datasets_encoding_within["test"]["V1"][0][i_voxel, :],
        predicted_responses["V1"][0][i_voxel, :],
        s=3,
        alpha=0.5,
    )
    ax.set_aspect("equal")
    ax.set_title(f"voxel {i_voxel + 1}")

fig.supxlabel("true data")
fig.supylabel("reconstructed data")
fig.set_facecolor("w")

In [ ]:
fig, axes = plt.subplots(figsize=(6, 2.5), nrows=1, ncols=4, sharex=True, sharey=True)

for roi, ax in zip(ROIS, axes.flat, strict=True):
    noise_ceiling = pearson_r(
        torch.from_numpy(datasets_encoding_within["test"][roi][0].to_numpy()),
        torch.from_numpy(datasets_encoding_within["test"][roi][1].to_numpy()),
    )
    model_performance = pearson_r(
        torch.from_numpy(datasets_encoding_within["test"][roi][0].to_numpy()),
        torch.from_numpy(predicted_responses[roi][0].to_numpy()),
    )
    ax.scatter(
        noise_ceiling,
        model_performance,
        s=3,
    )
    ax.set_xlim(left=0, right=1)
    ax.set_ylim(bottom=0, top=1)
    ax.axline([0, 0], [1, 1], c="gray", ls="-")
    ax.set_title(roi)

fig.supxlabel("noise ceiling\nr(repetition 1, repetition 2)")
fig.supylabel("model performance\nr(repetition 1, predicted repetition 1)", ha="center")

In [ ]:
func = functools.partial(
    compute_spectra_with_n_fold_cross_validation,
    n_folds=8,
    n_permutations=5_000,
    metric="covariance",
)

spectra = {"within": [], "between": []}
spectra_unbinned = {"within": [], "between": []}
for roi in tqdm(ROIS, desc="region of interest", leave=False):
    spectra_ = func(
        x_train=datasets_encoding_within["test"][roi][0],
        y_train=datasets_encoding_within["test"][roi][1],
        x_test=datasets_encoding_within["test"][roi][0],
        y_test=datasets_encoding_within["test"][roi][1],
    )
    spectra_unbinned["within"].append(
        spectra_["covariance"].expand_dims({
            "region of interest": [roi],
            "comparison": ["within-subject"],
        }),
    )
    spectra_ = bin_data(
        spectra_,
        bin_edges={"component": bin_edges},
        bin_centers={"component": bin_centers},
        dim="rank",
    ).expand_dims({
        "region of interest": [roi],
        "comparison": ["within-subject"],
    })
    spectra["within"].append(spectra_)

    spectra_ = func(
        x_train=datasets_encoding_within["test"][roi][0],
        y_train=predicted_responses[roi][1],
        x_test=datasets_encoding_within["test"][roi][0],
        y_test=predicted_responses[roi][1],
    )
    spectra_unbinned["within"].append(
        spectra_["covariance"].expand_dims({
            "region of interest": [roi],
            "comparison": ["brain-model"],
        }),
    )
    spectra_ = bin_data(
        spectra_,
        bin_edges={"component": bin_edges},
        bin_centers={"component": bin_centers},
        dim="rank",
    ).expand_dims({
        "region of interest": [roi],
        "comparison": ["brain-model"],
    })
    spectra["within"].append(spectra_)

spectra_unbinned["within"] = xr.merge(spectra_unbinned["within"])
spectra["within"] = xr.merge(spectra["within"])

In [ ]:
fig, axes = plt.subplots(
    ncols=len(ROIS),
    nrows=1,
    figsize=(6, 3),
    sharex=True,
    sharey=True,
)
for ax, roi in zip(axes.flat, ROIS, strict=True):
    plot_spectra(
        spectra["within"].sel({"region of interest": roi}),
        ax=ax,
        hue="comparison",
        metric="covariance",
        hue_order=["brain-model", "within-subject"],
        hide_insignificant=True,
        null_quantile=0.999,
        palette=["mediumvioletred", "dimgray"],
    )
    ax.set_title(roi)

ax = axes[0]
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-8, top=1e-1)
ax.set_xlim(left=1, right=2e3)
ax.set_xticks([1, 1e1, 1e2, 1e3])

ytick_exponents = list(range(-8, 0))
ax.set_yticks(
    [10**exponent for exponent in ytick_exponents],
    labels=[
        mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
        for exponent in ytick_exponents
    ],
)
ax.legend(borderaxespad=0.25, borderpad=0.25, handletextpad=0.05)

fig.supxlabel("rank", x=0.55)
fig.supylabel("covariance", y=0.53)
fig.suptitle(
    "shared variance between brain responses on one repetition and brain responses\non a different repetition reconstructed using Gabor filter bank features",
    y=1.05,
)
fig.set_facecolor("w")
# sns.move_legend(axes[0, 0], loc="center right", bbox_to_anchor=(6, 0.75))

# save_figure(fig_cumulative, filepath=FIGURES_HOME / "gabor-filter-bank-reconstructed.pdf")


In [ ]:
fig, axes = plt.subplots(
    ncols=len(ROIS),
    nrows=1,
    figsize=(6, 2.75),
    sharex=True,
    sharey=True,
)
for ax, roi in zip(axes.flat, ROIS, strict=True):
    spectra_ = spectra_unbinned["within"]["covariance"].sel({"region of interest": roi})

    # this do be kinda sus, but this is what Stringer et al. (2019) does *confused shrug*
    spectra_ = spectra_.where(spectra_ > 0, 0)
    spectra_ /= spectra_.sum("component")

    sns.lineplot(
        ax=ax,
        data=spectra_.cumsum("component").to_dataframe().reset_index(),
        x="component",
        y="covariance",
        hue="comparison",
        hue_order=["brain-model", "within-subject"],
        palette=["mediumvioletred", "dimgray"],
        legend=roi == "V1",
        errorbar="sd",
    )
    ax.set_title(roi)
    ax.set_xlabel("")

ax = axes[0]
ax.set_xscale("log")
ax.set_ylim(bottom=0)
ax.set_xlim(left=1, right=2e3)
ax.set_xticks([1, 1e1, 1e2, 1e3])
ax.set_ylabel("cumulative covariance\n(normalized)")
ax.legend()

fig.supxlabel("rank", x=0.55, y=0.05)
fig.set_facecolor("w")

# save_figure(fig, filepath=FIGURES_HOME / "gabor-model.pdf")

### between-subject

In [ ]:
n_train = len(shared_stimuli) // 2
stimuli = {
    "train": set(sorted(shared_stimuli)[:n_train]),
    "test": set(sorted(shared_stimuli)[n_train:]),
}

datasets_encoding_between = {
    subset: {
        roi: {
            subject: split_by_repetition(
                filter_by_stimulus(
                    dataset[subject],
                    stimuli=stimuli_,
                ),
                n_repetitions=3,
            )
            for subject in range(2)
        }
        for roi, dataset in datasets.items()
    }
    for subset, stimuli_ in stimuli.items()
}

gabor_model_responses_between = {
    subset: filter_by_stimulus(gabor_model_responses, stimuli=stimuli_)
    for subset, stimuli_ in stimuli.items()
}

In [ ]:
z_score = False

predicted_responses = {}
for roi in tqdm(ROIS, desc="region of interest", leave=False):
    predicted_responses[roi] = {}
    for repetition in tqdm((0, 1), desc="repetition", leave=False):
        linear_regression = fit_linear_ridge_regression(
            gabor_model_responses_between["train"],
            datasets_encoding_between["train"][roi][1][repetition],
            alphas=ALPHAS,
        )

        z_ = datasets_encoding_between["test"][roi][1][repetition].copy()
        z_ = z_.rename(f"{z_.name}.predicted.ridge")
        z_.data = linear_regression.predict(
            gabor_model_responses_between["test"].to_numpy(),
        )

        if z_score:
            z_ = (z_ - z_.mean("presentation")) / z_.std("presentation")
            z_ = z_.rename(f"{z_.name}.z_score=True")

        predicted_responses[roi][repetition] = z_.astype(np.float32)


In [ ]:
func = functools.partial(
    compute_spectra_with_n_fold_cross_validation,
    n_folds=8,
    n_permutations=5_000,
    metric="covariance",
)

for roi in tqdm(ROIS, desc="region of interest", leave=False):
    spectra_ = func(
        x_train=datasets_encoding_between["test"][roi][0][0],
        y_train=datasets_encoding_between["test"][roi][1][1],
        x_test=datasets_encoding_between["test"][roi][0][0],
        y_test=datasets_encoding_between["test"][roi][1][1],
    )
    spectra_unbinned["between"].append(
        spectra_["covariance"].expand_dims({
            "region of interest": [roi],
            "comparison": ["between-subject"],
        }),
    )
    spectra_ = bin_data(
        spectra_,
        bin_edges={"component": bin_edges},
        bin_centers={"component": bin_centers},
        dim="rank",
    ).expand_dims({
        "region of interest": [roi],
        "comparison": ["between-subject"],
    })
    spectra["between"].append(spectra_)

    spectra_ = func(
        x_train=datasets_encoding_between["test"][roi][0][0],
        y_train=predicted_responses[roi][1],
        x_test=datasets_encoding_between["test"][roi][0][0],
        y_test=predicted_responses[roi][1],
    )
    spectra_unbinned["between"].append(
        spectra_["covariance"].expand_dims({
            "region of interest": [roi],
            "comparison": ["brain-model"],
        }),
    )
    spectra_ = bin_data(
        spectra_,
        bin_edges={"component": bin_edges},
        bin_centers={"component": bin_centers},
        dim="rank",
    ).expand_dims({
        "region of interest": [roi],
        "comparison": ["brain-model"],
    })
    spectra["between"].append(spectra_)

spectra_unbinned["between"] = xr.merge(spectra_unbinned["between"])
spectra["between"] = xr.merge(spectra["between"])

In [ ]:
fig, axes = plt.subplots(
    ncols=len(ROIS),
    nrows=1,
    figsize=(6, 2.75),
    sharex=True,
    sharey=True,
)
for ax, roi in zip(axes.flat, ROIS, strict=True):
    spectra_ = spectra_unbinned["between"]["covariance"].sel({
        "region of interest": roi
    })

    # this do be kinda sus, but this is what Stringer et al. (2019) does *confused shrug*
    spectra_ = spectra_.where(spectra_ > 0, 0)
    spectra_ /= spectra_.sum("component")

    sns.lineplot(
        ax=ax,
        data=spectra_.cumsum("component").to_dataframe().reset_index(),
        x="component",
        y="covariance",
        hue="comparison",
        hue_order=["brain-model", "between-subject"],
        palette=["mediumvioletred", "dimgray"],
        legend=roi == "V1",
        errorbar="sd",
    )
    ax.set_title(roi)
    ax.set_xlabel("")

ax = axes[0]
ax.set_xscale("log")
ax.set_ylim(bottom=0)
ax.set_xlim(left=1, right=1e3)
ax.set_xticks([1, 1e1, 1e2, 1e3])
ax.set_ylabel("cumulative covariance\n(normalized)")

ax.legend()

fig.supxlabel("rank", x=0.55, y=0.05)
fig.set_facecolor("w")

# save_figure(fig, filepath=FIGURES_HOME / "gabor-model.pdf")

In [ ]:
colors = {
    "within-subject": sns.color_palette("crest", n_colors=1)[0],
    "between-subject": sns.color_palette("flare", n_colors=1)[0],
    "brain-model": "dimgray",
}


fig, axes = plt.subplots(
    ncols=len(ROIS),
    nrows=2,
    figsize=(6, 4),
    sharex=True,
    sharey=True,
)
for i_roi, roi in enumerate(ROIS):
    for i_row, type_ in enumerate(("within", "between")):
        spectra_ = spectra_unbinned[type_]["covariance"].sel({
            "region of interest": roi
        })

        # this do be kinda sus, but this is what Stringer et al. (2019) does *confused shrug*
        spectra_ = spectra_.where(spectra_ > 0, 0)
        spectra_ /= spectra_.sum("component")

        ax = axes[i_row, i_roi]
        hue_order = ["brain-model", f"{type_}-subject"]
        sns.lineplot(
            ax=ax,
            data=spectra_.cumsum("component").to_dataframe().reset_index(),
            x="component",
            y="covariance",
            hue="comparison",
            hue_order=hue_order,
            palette=[colors[hue_] for hue_ in hue_order],
            legend=roi == "V1",
            errorbar="sd",
        )
        if i_row == 0:
            ax.set_title(roi)
        ax.set_xlabel("")
        ax.set_ylabel("")
        if i_roi == 0:
            ax.legend()

ax = axes[0, 0]
ax.set_xscale("log")
ax.set_ylim(bottom=0)
ax.set_xlim(left=1, right=1e3)
ax.set_xticks([1, 1e1, 1e2, 1e3])

fig.supylabel("cumulative covariance\n(normalized)", ha="center", y=0.54, x=0.05)
fig.supxlabel("rank", x=0.55, y=0.05)
save_figure(fig, filepath=FIGURES_HOME / "gabor-model.pdf")

## Mick's suggestion: train on (X_train, Y_train), test on (X_test_, Y_test_reconstructed)

In [ ]:
func = functools.partial(
    compute_spectra_with_n_fold_cross_validation,
    n_folds=8,
    n_permutations=5_000,
    metric="covariance",
)

spectra = []

for roi in tqdm(ROIS, desc="region of interest", leave=False):
    cross_decomposition = CrossDecomposition(randomized=False)
    cross_decomposition.fit(
        datasets_encoding["train"][roi][0],
        datasets_encoding["train"][roi][1],
    )
    spectrum = cross_decomposition.compute_spectrum(
        datasets_encoding["test"][roi][0],
        datasets_encoding["test"][roi][1],
        metric="covariance",
    ).expand_dims(comparison=["within-subject"])

    spectrum_permuted = cross_decomposition.compute_permuted_spectra(
        datasets_encoding["test"][roi][0],
        datasets_encoding["test"][roi][1],
        metric="covariance",
        n_permutations=5_000,
    ).expand_dims(comparison=["within-subject"])

    spectrum_cross = cross_decomposition.compute_spectrum(
        datasets_encoding["test"][roi][0],
        predicted_responses[roi][1],
        metric="covariance",
    ).expand_dims(comparison=["brain-model"])

    spectrum_cross_permuted = cross_decomposition.compute_permuted_spectra(
        datasets_encoding["test"][roi][0],
        predicted_responses[roi][1],
        metric="covariance",
        n_permutations=5_000,
    ).expand_dims(comparison=["brain-model"])

    spectra_ = bin_data(
        xr.merge([
            spectrum,
            spectrum_cross,
            spectrum_permuted,
            spectrum_cross_permuted,
        ]),
        bin_edges={"component": bin_edges},
        bin_centers={"component": bin_centers},
        dim="rank",
    ).expand_dims({
        "region of interest": [roi],
    })
    spectra.append(spectra_)

spectra = xr.merge(spectra)

In [ ]:
fig, axes = plt.subplots(
    ncols=len(ROIS) // 2,
    nrows=2,
    figsize=(6, 4),
    sharex=True,
    sharey=True,
)
for ax, roi in zip(axes.flat, ROIS, strict=True):
    sns.lineplot(
        ax=ax,
        data=spectra.sel({"region of interest": roi}).to_dataframe().reset_index(),
        x="rank",
        y="covariance",
        hue="comparison",
        legend=roi == "V1",
        ls="None",
        marker="o",
    )
    ax.set_title(roi)
    ax.set_xlabel("")
    ax.set_ylabel("")

    # plot_spectra(
    #     spectra.sel({"region of interest": roi}),
    #     ax=ax,
    #     hue="comparison",
    #     metric="covariance",
    #     hue_order=["brain-model", "within-subject"],
    #     hide_insignificant=True,
    #     null_quantile=0.999,
    #     palette=["mediumvioletred", "dimgray"],
    # )
    # ax.set_title(roi)

ax = axes[0, 0]
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(bottom=1e-5, top=1e3)
ax.set_xlim(left=1, right=2e3)
ax.set_xticks([1, 1e1, 1e2, 1e3])

ytick_exponents = list(range(-5, 4))
ax.set_yticks(
    [10**exponent for exponent in ytick_exponents],
    labels=[
        mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
        for exponent in ytick_exponents
    ],
)
ax.legend(borderaxespad=0.25, borderpad=0.25, handletextpad=0.05)

fig.supxlabel("rank", x=0.55)
fig.supylabel("covariance", y=0.53)
fig.suptitle(
    "learn rotations of real data, test on real vs predicted data",
)
fig.set_facecolor("w")
sns.move_legend(axes[0, 0], loc="center right", bbox_to_anchor=(6, 0.75))